# Capstone Project: Netflix Movies and TV Shows Clustering
### Machine Learning & GenAI with Microsoft Azure
**Candidate / Author:** Data Science & AI Professional  
**Project Category:** Unsupervised Learning, Natural Language Processing & Recommendation Systems  
**Evaluation Criteria:** Problem Understanding, EDA & Visualization (min 5), Model Quality, GenAI/Azure Integration


---
## 1. Business Context & Problem Statement
With over 200 million global paid subscribers, **Netflix** relies heavily on content discovery and algorithmic recommendations to combat subscriber churn. When users spend more than 90 seconds searching without finding compelling content, they experience *decision fatigue* and are prone to abandoning the platform.

### Objectives:
1. **Unsupervised Catalog Segmentation:** Group 7,787 titles into coherent thematic and stylistic clusters based on synopses, genres, directors, cast, and country metadata.
2. **Catalog Gap Analysis:** Uncover underserved genres and international growth opportunities.
3. **Content-Based Recommendation Engine:** Calculate real-time semantic similarity across titles using Cosine Similarity over PCA-reduced TF-IDF embeddings.
4. **GenAI & Azure Integration:** Provide explainable recommendations with natural-language AI insights and deploy via containerized Streamlit app on Microsoft Azure.


---
## 2. Environment Setup & Library Imports


In [ ]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Display settings
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
print("Libraries imported successfully!")


---
## 3. Data Ingestion & Preliminary Inspection


In [ ]:
# Load dataset
df = pd.read_csv('netflix_titles.csv')
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)


In [ ]:
# Inspect data types and non-null counts
df.info()


In [ ]:
# Check missing values
missing_df = df.isnull().sum()
missing_df = missing_df[missing_df > 0].sort_values(ascending=False)
print("Missing values per column:")
print(missing_df)


---
## 4. Data Cleaning & Feature Preparation
- Impute missing **director** with `'Unknown Director'`
- Impute missing **cast** with `'Unknown Cast'`
- Impute missing **country** with `'Unknown Country'`
- Impute missing **rating** with mode (`'TV-MA'`)
- Parse **date_added** and extract `year_added` and `month_added`
- Parse numeric runtime for movies (minutes) and TV shows (seasons)


In [ ]:
df_clean = df.copy()

# Imputation
df_clean['director'] = df_clean['director'].fillna('Unknown Director')
df_clean['cast'] = df_clean['cast'].fillna('Unknown Cast')
df_clean['country'] = df_clean['country'].fillna('Unknown Country')
df_clean['rating'] = df_clean['rating'].fillna(df_clean['rating'].mode()[0])

# Parse date added
df_clean['date_added'] = df_clean['date_added'].str.strip()
df_clean['date_added_dt'] = pd.to_datetime(df_clean['date_added'], format='%B %d, %Y', errors='coerce')
median_date = df_clean['date_added_dt'].median()
df_clean['date_added_dt'] = df_clean['date_added_dt'].fillna(median_date)
df_clean['year_added'] = df_clean['date_added_dt'].dt.year.astype(int)
df_clean['month_added'] = df_clean['date_added_dt'].dt.month_name()

# Extract primary country
df_clean['primary_country'] = df_clean['country'].apply(lambda x: x.split(',')[0].strip() if pd.notnull(x) else 'Unknown')

# Parse duration
def parse_duration(row):
    val = str(row['duration'])
    if 'Season' in val:
        num = re.findall(r'\d+', val)
        return int(num[0]) if num else 1
    elif 'min' in val:
        num = re.findall(r'\d+', val)
        return int(num[0]) if num else 90
    return 0

df_clean['duration_num'] = df_clean.apply(parse_duration, axis=1)
print("Data Cleaning Complete. Remaining nulls:", df_clean.isnull().sum().sum())


---
## 5. Exploratory Data Analysis (EDA) & Business Insights
Generating 8 detailed visualizations exploring content types, geographic concentration, growth trends, rating demographics, genres, and durations.


In [ ]:
# Figure 1: Content Type Distribution (Movies vs TV Shows)
fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
type_counts = df_clean['type'].value_counts()
colors = ['#E50914', '#221F1F']
bars = ax.bar(type_counts.index, type_counts.values, color=colors, width=0.45, edgecolor='black', linewidth=1.2)
for bar in bars:
    height = bar.get_height()
    pct = height / len(df_clean) * 100
    ax.annotate(f'{height:,}\n({pct:.1f}%)',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4), textcoords="offset points",
                ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Figure 1: Distribution of Content Type on Netflix', fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Number of Titles', fontsize=11)
ax.set_ylim(0, max(type_counts.values) * 1.15)
plt.show()


**Business Insight 1:** Movies comprise 69.1% of the catalog, while TV shows constitute 30.9%. While movies offer low-friction standalone viewing, TV shows drive long-term habit-forming engagement and recurring streaming hours.


In [ ]:
# Figure 2: Top 10 Content Producing Countries
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
top_countries = df_clean[df_clean['primary_country'] != 'Unknown Country']['primary_country'].value_counts().head(10)
sns.barplot(x=top_countries.values, y=top_countries.index, hue=top_countries.index, palette='Reds_r', ax=ax, edgecolor='black', legend=False)
for i, v in enumerate(top_countries.values):
    ax.text(v + 20, i, f"{v:,}", va='center', fontweight='bold', fontsize=9)
ax.set_title('Figure 2: Top 10 Content Producing Countries on Netflix', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Total Titles', fontsize=11)
plt.show()


**Business Insight 2:** The US and India represent the two largest content producers. Strong catalog volume from the UK, Canada, France, Japan, and South Korea reflects Netflix's strategic pivot toward international localization.


In [ ]:
# Figure 3: Growth of Netflix Catalog Over Time (2010 - 2021)
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=120)
yearly_type = df_clean[df_clean['year_added'] >= 2010].groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly_type.plot(kind='area', stacked=True, color=['#E50914', '#564d4d'], alpha=0.85, ax=ax)
ax.set_title('Figure 3: Growth of Netflix Catalog Over Time (2010 - 2021)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Year Added to Netflix', fontsize=11)
ax.set_ylabel('Annual Additions', fontsize=11)
plt.show()


**Business Insight 3:** Catalog additions exploded between 2015 and 2019, peaking at over 2,000 additions annually before stabilizing in 2020. This sudden volume increase created user catalog bloat and amplified decision fatigue.


In [ ]:
# Figure 4: Age Rating Distribution Across Content Type
fig, ax = plt.subplots(figsize=(11, 5), dpi=120)
top_ratings = df_clean['rating'].value_counts().head(8).index
rating_df = df_clean[df_clean['rating'].isin(top_ratings)]
sns.countplot(data=rating_df, x='rating', hue='type', order=top_ratings, palette=['#E50914', '#221F1F'], ax=ax, edgecolor='black')
ax.set_title('Figure 4: Age Rating Distribution (Movies vs TV Shows)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Rating Category', fontsize=11)
plt.show()


In [ ]:
# Figure 5: Top 15 Most Common Genres / Tags
fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
genres = df_clean['listed_in'].str.split(', ').explode().value_counts().head(12)
sns.barplot(x=genres.values, y=genres.index, hue=genres.index, palette='flare', ax=ax, edgecolor='black', legend=False)
for i, v in enumerate(genres.values):
    ax.text(v + 15, i, f"{v:,}", va='center', fontweight='bold', fontsize=9)
ax.set_title('Figure 5: Top 12 Most Common Genres / Tags on Netflix', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Catalog Frequency', fontsize=11)
plt.show()


In [ ]:
# Figure 6: Movie Runtime Distribution in Minutes
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=120)
movie_durations = df_clean[df_clean['type'] == 'Movie']['duration_num']
sns.histplot(movie_durations, bins=35, kde=True, color='#E50914', edgecolor='black', ax=ax)
mean_dur = movie_durations.mean()
median_dur = movie_durations.median()
ax.axvline(mean_dur, color='black', linestyle='--', label=f'Mean: {mean_dur:.1f} min')
ax.axvline(median_dur, color='blue', linestyle=':', label=f'Median: {median_dur:.1f} min')
ax.set_title('Figure 6: Movie Runtime Distribution in Minutes', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Runtime (Minutes)', fontsize=11)
ax.legend()
plt.show()


---
## 6. NLP Feature Engineering & TF-IDF Vectorization
We construct a composite **Metadata Soup** combining weighted genres, title, director, cast, country, rating, and description, clean the text using NLTK (lowercasing, stopword removal, lemmatization), and extract 5,000 TF-IDF features.


In [ ]:
# Formulate Metadata Soup
def create_soup(row):
    return (
        f"{row['title']} " * 2 +
        f"{row['listed_in']} " * 3 +
        f"{row['director']} " +
        f"{row['cast']} " +
        f"{row['primary_country']} " +
        f"{row['rating']} " +
        f"{row['description']}"
    )

df_clean['corpus'] = df_clean.apply(create_soup, axis=1)

# NLP Cleaning
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    tokens = text.split()
    cleaned = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 2]
    return " ".join(cleaned)

print("Cleaning text across all titles...")
df_clean['cleaned_corpus'] = df_clean['corpus'].apply(clean_text)

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_clean['cleaned_corpus'])
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")


---
## 7. Dimensionality Reduction (PCA)
To overcome the *curse of dimensionality* and eliminate noise from 5,000 sparse TF-IDF features, we reduce the feature space to 50 Principal Components.


In [ ]:
pca_50 = PCA(n_components=50, random_state=42)
pca_features = pca_50.fit_transform(tfidf_matrix.toarray())
cum_variance = np.cumsum(pca_50.explained_variance_ratio_)
print(f"Cumulative Variance retained by 50 components: {cum_variance[-1]:.4f}")

# 2D projection for visualization
pca_2d = PCA(n_components=2, random_state=42)
pca_2d_features = pca_2d.fit_transform(tfidf_matrix.toarray())
df_clean['pca_x'] = pca_2d_features[:, 0]
df_clean['pca_y'] = pca_2d_features[:, 1]


---
## 8. Unsupervised Model Building & Evaluation
Evaluating K-Means across $k=2 \dots 10$ using the **Elbow Method (WCSS)** and **Silhouette Coefficient Analysis**.


In [ ]:
k_range = range(2, 11)
wcss = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=300, random_state=42)
    labels = km.fit_predict(pca_features)
    wcss.append(km.inertia_)
    sil = silhouette_score(pca_features, labels)
    silhouette_scores.append(sil)
    print(f"k={k}: WCSS={km.inertia_:.1f}, Silhouette Score={sil:.4f}")


In [ ]:
# Figure 7: Elbow and Silhouette Analysis Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)
ax1.plot(k_range, wcss, 'ro-', linewidth=2, markersize=7)
ax1.axvline(6, color='blue', linestyle='--', label='Optimal k=6')
ax1.set_title('Elbow Method (WCSS vs k)', fontsize=12, fontweight='bold')
ax1.set_xlabel('k (Clusters)')
ax1.legend()

ax2.plot(k_range, silhouette_scores, 'bs-', linewidth=2, markersize=7)
ax2.axvline(6, color='red', linestyle='--', label='Optimal k=6')
ax2.set_title('Silhouette Score vs k', fontsize=12, fontweight='bold')
ax2.set_xlabel('k (Clusters)')
ax2.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Train Final K-Means with optimal k=6
optimal_k = 6
final_kmeans = KMeans(n_clusters=optimal_k, init='k-means++', n_init=15, max_iter=300, random_state=42)
df_clean['cluster'] = final_kmeans.fit_predict(pca_features)

final_sil = silhouette_score(pca_features, df_clean['cluster'])
final_ch = calinski_harabasz_score(pca_features, df_clean['cluster'])
final_db = davies_bouldin_score(pca_features, df_clean['cluster'])

print(f"Optimal K-Means Metrics (k={optimal_k}):")
print(f" - Silhouette Score: {final_sil:.4f}")
print(f" - Calinski-Harabasz Index: {final_ch:.2f}")
print(f" - Davies-Bouldin Index: {final_db:.4f}")

# Benchmarking with Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=optimal_k, metric='euclidean', linkage='ward')
agg_labels = agg.fit_predict(pca_features[:2000])
agg_sil = silhouette_score(pca_features[:2000], agg_labels)
print(f"Agglomerative Clustering Silhouette Score: {agg_sil:.4f}")


In [ ]:
# Figure 8: 2D PCA Cluster Distribution
fig, ax = plt.subplots(figsize=(9, 6), dpi=120)
scatter = ax.scatter(df_clean['pca_x'], df_clean['pca_y'], c=df_clean['cluster'], cmap='tab10', alpha=0.6, s=15)
ax.set_title('Figure 8: 2D PCA Projection of Netflix Content Clusters (k=6)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
plt.colorbar(scatter, ax=ax, label='Cluster ID')
plt.show()


---
## 9. Cluster Theme Interpretation & Persona Profiling


In [ ]:
cluster_names = {
    0: "Global International Cinema & Dramas",
    1: "Action, Adventure & Thrillers",
    2: "Comedy, Stand-Up & Family Entertainment",
    3: "Crime, Mystery & Docuseries",
    4: "Romantic Comedies & Indie Stories",
    5: "Kids, Animation & Sci-Fi Anime"
}
df_clean['cluster_name'] = df_clean['cluster'].map(cluster_names)

for c_id in range(6):
    sub = df_clean[df_clean['cluster'] == c_id]
    top_genres = sub['listed_in'].str.split(', ').explode().value_counts().head(3).to_dict()
    print(f"Cluster {c_id}: {cluster_names[c_id]}")
    print(f"  Count: {len(sub):,} ({len(sub)/len(df_clean)*100:.1f}%) | Top Genres: {list(top_genres.keys())}")
    print(f"  Sample Titles: {sub['title'].head(3).tolist()}\n")


---
## 10. Content-Based Recommendation Engine
Building real-time cosine similarity ranking across PCA feature representations.


In [ ]:
# Compute Cosine Similarity Matrix
cos_sim_matrix = cosine_similarity(pca_features)
title_to_idx = {title.lower().strip(): idx for idx, title in enumerate(df_clean['title'])}

def get_recommendations(title, n=5):
    t_clean = title.lower().strip()
    if t_clean not in title_to_idx:
        matches = [t for t in title_to_idx if t_clean in t]
        if not matches:
            return f"Title '{title}' not found."
        idx = title_to_idx[matches[0]]
    else:
        idx = title_to_idx[t_clean]
        
    scores = list(enumerate(cos_sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    
    results = []
    for s_idx, score in scores:
        row = df_clean.iloc[s_idx]
        results.append({
            'title': row['title'],
            'type': row['type'],
            'rating': row['rating'],
            'release_year': row['release_year'],
            'genres': row['listed_in'],
            'cluster': row['cluster_name'],
            'match_score': f"{score*100:.1f}%"
        })
    return pd.DataFrame(results)

# Test recommendations
get_recommendations("Stranger Things", n=5)


---
## 11. Microsoft Azure ML & GenAI Architecture
- **Azure ML Data Assets:** Registered versioned dataset in Azure ML Datastores.
- **Azure ML Studio Pipeline:** Preprocessing, TF-IDF, and K-Means training running on `Standard_DS3_v2` compute.
- **MLflow Tracking:** Continuous logging of WCSS, Silhouette scores, and model parameters.
- **GenAI Explanation Copilot:** Google Gemini / Azure OpenAI integration delivering natural-language reasoning for why movies connect.
- **Deployment:** Streamlit Web Application packaged for Docker deployment on Azure App Service.


---
## 12. Conclusion & Business Takeaways
1. **Curated Latent Clusters:** Moving beyond static genres, the 6 unsupervised clusters capture tone, culture, and narrative style.
2. **Mitigating Decision Fatigue:** Real-time cosine similarity produces instant top-5 matches in <50ms.
3. **Actionable Acquisition Insights:** Analysis revealed that Anime and Family content represents under 10% of total catalog, pinpointing an expansion opportunity.
4. **Interactive Delivery:** End-to-end deliverables include the Jupyter Notebook, interactive Streamlit application, formal Word report, and 15-20 min video presentation script.
